In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime
import uuid

from utils import (
    get_hdfs_base_uri,
    create_namespace,
    read_csv_raw,
    normalize_dataframe_columns,
    cast_all_columns_to_string,
    add_bronze_metadata,
    add_stable_bronze_id,
    add_record_hash,
    merge_iceberg_by_keys,
)

spark = SparkSession.builder.getOrCreate()

HDFS_BASE_URI = get_hdfs_base_uri()

CATALOG = "spark_catalog"
BRONZE_DB = "bronze"
BRONZE_TABLE = f"{CATALOG}.{BRONZE_DB}.beneficiarios"

RAW_PATH = f"{HDFS_BASE_URI}/dados/raw/ans/"
BRONZE_LOCATION = f"{HDFS_BASE_URI}/user/hive/warehouse/{BRONZE_DB}.db"

BATCH_ID = datetime.now().strftime("%Y%m%d%H%M%S") + "_" + uuid.uuid4().hex[:8]

create_namespace(
    spark=spark,
    catalog=CATALOG,
    database=BRONZE_DB,
    location=BRONZE_LOCATION,
)

df_raw = read_csv_raw(
    spark=spark,
    path=RAW_PATH,
    sep=";",
)

df_normalized = normalize_dataframe_columns(df_raw)
df_bronze = cast_all_columns_to_string(df_normalized)

bronze_id_columns = [
    "id_cmpt_movel",
    "cd_operadora",
    "cd_municipio",
    "cd_plano",
    "tp_sexo",
    "de_faixa_etaria",
    "de_faixa_etaria_reaj",
    "tipo_vinculo",
    "dt_carga",
]

payload_columns = df_bronze.columns

df_bronze = (
    df_bronze
    .transform(lambda df: add_bronze_metadata(
        df=df,
        source_system="ANS",
        batch_id=BATCH_ID,
    ))
    .transform(lambda df: add_stable_bronze_id(
        df=df,
        id_columns=bronze_id_columns,
        id_column_name="_bronze_id",
    ))
    .transform(lambda df: add_record_hash(
        df=df,
        payload_columns=payload_columns,
        hash_column_name="_record_hash",
    ))
)

df_bronze = df_bronze.dropDuplicates(["_bronze_id"])

merge_iceberg_by_keys(
    spark=spark,
    source_df=df_bronze,
    target_table=BRONZE_TABLE,
    key_columns=["_bronze_id"],
    source_view="vw_bronze_beneficiarios_incremental",
    compare_hash_column="_record_hash",
    non_update_columns=["_ingested_at"],
)